<a href="https://colab.research.google.com/github/kunphat510214-netizen/project6/blob/Step-4/%E0%B8%82%E0%B8%B1%E0%B9%89%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88_4_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#ขั้นที่ 2
import random
import sqlite3
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print


class Member:
    def __init__(self, member_id, name, points=0):
        self.member_id = member_id
        self.name = name
        self.points = points

    def add_points(self, cups=1):
        self.points += cups
        return self.points


class DrinkOrder:
    BASE_PRICES = {
        "อเมริกาโน่": 45, "ลาเต้": 50, "คาปูชิโน่": 50,
        "เอสเพรสโซ่": 50, "มอคค่า": 50, "ชาเขียว": 45,
        "ชาไทย": 45, "มัทฉะลาเต้": 50, "ช็อกโกแลต": 50,
        "นมสด": 40, "สตรอว์เบอร์รี่โซดา": 40,
        "บลูฮาวายโซดา": 40, "ยูซุโซดา": 40,
    }
    TYPE_PRICES = {"ร้อน": 0, "เย็น": 5, "ปั่น": 10}
    SIZE_PRICES = {"S": 0, "M": 10, "L": 20}
    TOPPING_PRICES = {"ไม่มี": 0, "ไข่มุก": 5, "วิปครีม": 15, "ครีมชีส": 20, "เจลลี่": 5}
    PAYMENT_METHODS = ("เงินสด", "QR Code", "บัตร")

    def __init__(self, order_id, customer_name, menu_name, drink_type, size,
                 topping, sweetness_level, receive_type, member_id=None):
        self.order_id = order_id
        self.queue_no = f"Q{order_id:03d}"
        self.customer_name = customer_name
        self.menu_name = menu_name
        self.drink_type = drink_type
        self.size = size
        self.topping = topping
        self.sweetness_level = sweetness_level
        self.receive_type = receive_type
        self.member_id = member_id
        self.payment_method = None
        self.payment_status = "ยังไม่ชำระ"
        self.status = "รอชำระเงิน"
        self.wait_minutes = 0

    def calculate_subtotal(self):
        return (self.BASE_PRICES[self.menu_name]
                + self.TYPE_PRICES[self.drink_type]
                + self.SIZE_PRICES[self.size]
                + self.TOPPING_PRICES[self.topping])

    def calculate_price(self):
        # สมาชิกได้รับส่วนลด 5%
        discount_rate = 0.05 if self.member_id else 0
        return round(self.calculate_subtotal() * (1 - discount_rate), 2)

    def confirm_payment(self, payment_method):
        if payment_method not in self.PAYMENT_METHODS:
            raise ValueError("ช่องทางชำระเงินไม่ถูกต้อง")
        self.payment_method = payment_method
        self.payment_status = "ชำระแล้ว"
        self.status = "รอดำเนินการ"

    def receipt_text(self):
        return (f"ใบเสร็จ R{self.order_id:05d} | {self.queue_no} | "
                f"{self.menu_name} {self.drink_type} {self.size} | "
                f"{self.calculate_price():,.2f} บาท | {self.payment_method}")

    def sticker_text(self):
        return (f"{self.queue_no} {self.menu_name}/{self.drink_type}/{self.size} "
                f"หวาน {self.sweetness_level} ท็อปปิ้ง {self.topping} ({self.receive_type})")

    def to_record(self):
        return {
            "order_id": self.order_id,
            "queue_no": self.queue_no,
            "customer_name": self.customer_name,
            "member_id": self.member_id,
            "menu_name": self.menu_name,
            "drink_type": self.drink_type,
            "size": self.size,
            "topping": self.topping,
            "sweetness_level": self.sweetness_level,
            "receive_type": self.receive_type,
            "payment_method": self.payment_method,
            "subtotal": self.calculate_subtotal(),
            "price": self.calculate_price(),
            "wait_minutes": self.wait_minutes,
            "status": self.status,
        }


class QueueSystem:
    def __init__(self):
        self.active_orders = []
        self.events = []

    def _log(self, order, event):
        self.events.append({
            "event_id": len(self.events) + 1,
            "order_id": order.order_id,
            "event": event,
            "status": order.status,
        })

    def send_to_kds(self, order):
        if order.payment_status != "ชำระแล้ว":
            raise ValueError("ต้องยืนยันการชำระเงินก่อนส่งเข้า KDS")
        self.active_orders.append(order)
        order.status = "รอดำเนินการ"
        self._log(order, "ส่งเข้า KDS")

    def start_preparing(self, order):
        order.status = "กำลังทำ"
        self._log(order, "บาริสต้ารับออเดอร์")

    def call_queue(self, order, announce=False):
        order.status = "พร้อมรับ"
        self._log(order, "เรียกคิว")
        message = f"🔔 คิว {order.queue_no} พร้อมรับที่เคาน์เตอร์"
        if announce:
            print(message, "(เสียงเรียก)")
        return message

    def complete_order(self, order):
        order.status = "เสร็จสิ้น"
        self._log(order, "ส่งมอบสำเร็จ")
        self.active_orders = [o for o in self.active_orders if o.order_id != order.order_id]

In [3]:
# ขั้นที่ 3: เขียน functions ช่วยงาน
import random

def generate_customer_name():
    first_names = ["เจสซี่", "ลีโอ", "มายา", "อเล็กซ์", "คลาร่า", "ลูคัส", "นิโคล"]
    last_names = ["รักเรียน", "ใจดี", "สายทอง", "รุ่งเรือง", "มั่นคง", "วงศ์สว่าง", "เจริญพร"]
    return f"{random.choice(first_names)} {random.choice(last_names)}"


def random_sweetness_level(levels=("0%", "25%", "50%", "75%", "100%")):
    return random.choice(levels)


def build_random_order(order_id, member=None):
    return DrinkOrder(
        order_id=order_id,
        customer_name=member.name if member else generate_customer_name(),
        menu_name=random.choice(list(DrinkOrder.BASE_PRICES)),
        drink_type=random.choice(list(DrinkOrder.TYPE_PRICES)),
        size=random.choice(list(DrinkOrder.SIZE_PRICES)),
        topping=random.choice(list(DrinkOrder.TOPPING_PRICES)),
        sweetness_level=random_sweetness_level(),
        receive_type=random.choice(["ทานที่ร้าน", "กลับบ้าน"]),
        member_id=member.member_id if member else None,
    )


demo_member = Member("M001", "มายา ใจดี", points=4)
demo_order = DrinkOrder(1, demo_member.name, "ลาเต้", "เย็น", "M", "วิปครีม",
                        "50%", "กลับบ้าน", demo_member.member_id)
demo_queue = QueueSystem()

print("1-3) รับลูกค้าและตัวเลือก:", demo_order.menu_name, demo_order.size,
      demo_order.topping, demo_order.sweetness_level, demo_order.receive_type,
      "สมาชิก" if demo_order.member_id else "ไม่เป็นสมาชิก")
print("4) สรุปราคา:", demo_order.calculate_price(), "บาท")
demo_order.confirm_payment("QR Code")
demo_member.add_points()
print("5) ชำระเงิน:", demo_order.payment_status, "ด้วย", demo_order.payment_method)
print("6)", demo_order.receipt_text())
print("   Sticker:", demo_order.sticker_text())
demo_queue.send_to_kds(demo_order)
print("7) KDS:", demo_order.status)
demo_queue.start_preparing(demo_order)
demo_queue.call_queue(demo_order, announce=True)
print("8-9) หน้าจอแสดง:", demo_order.queue_no, demo_order.status)
demo_queue.complete_order(demo_order)
print("10) สถานะ:", demo_order.status, "| คิวคงเหลือ:", len(demo_queue.active_orders))





1-3) รับลูกค้าและตัวเลือก: ลาเต้ M วิปครีม 50% กลับบ้าน สมาชิก
4) สรุปราคา: 76.0 บาท
5) ชำระเงิน: ชำระแล้ว ด้วย QR Code
6) ใบเสร็จ R00001 | Q001 | ลาเต้ เย็น M | 76.00 บาท | QR Code
   Sticker: Q001 ลาเต้/เย็น/M หวาน 50% ท็อปปิ้ง วิปครีม (กลับบ้าน)
7) KDS: รอดำเนินการ
🔔 คิว Q001 พร้อมรับที่เคาน์เตอร์ (เสียงเรียก)
8-9) หน้าจอแสดง: Q001 พร้อมรับ
10) สถานะ: เสร็จสิ้น | คิวคงเหลือ: 0


In [6]:
# ขั้นที่ 4: จำลองธุรกรรมทีละรายการด้วย loop อย่างน้อย 300 รายการ
random.seed(612104)
members = {
    f"M{i:03d}": Member(f"M{i:03d}", generate_customer_name(), random.randint(0, 9))
    for i in range(1, 61)
}

#สร้างระบบจัดการคิว และสร้างลิสต์สำหรับเก็บข้อมูลออเดอร์ทั้งหมด
queue_system = QueueSystem()
orders = []

#วนลูปเพื่อจำลองธุรกรรมจำนวน 300 รายการ
for i in range(1, 301):

  #สุ่มเลือกสมาชิก โดยมีโอกาส 40% ที่ออเดอร์จะเป็นของสมาชิก
    member = random.choice(list(members.values())) if random.random() < 0.40 else None

    #สร้างออเดอร์แบบสุ่ม พร้อมระบุหมายเลขออเดอร์และสมาชิก
    order = build_random_order(i, member)

    #ยืนยันการชำระเงินด้วยวิธีการชำระเงินแบบสุ่ม
    order.confirm_payment(random.choice(DrinkOrder.PAYMENT_METHODS))

    #หากเป็นสมาชิก ให้เพิ่มคะแนนสะสม
    if member:
        member.add_points()

   #ส่งออเดอร์เข้าสู่ระบบครัว (KDS)
    queue_system.send_to_kds(order)

    #เริ่มกระบวนการเตรียมเครื่องดื่ม
    queue_system.start_preparing(order)

    #สุ่มเวลารอรับเครื่องดื่มระหว่าง 2-18 นาที
    order.wait_minutes = random.randint(2, 18)

    #เรียกลูกค้าตามคิว
    queue_system.call_queue(order)

    #เปลี่ยนสถานะออเดอร์เป็นเสร็จสมบูรณ์
    queue_system.complete_order(order)

    #เก็บออเดอร์ที่ดำเนินการสร้างแล้วไว้ในลิสต์
    orders.append(order)

#ตรวจสอบว่ามีออเดอร์ครบ 300 รายการ
assert len(orders) == 300

#ตรวจสอบว่าไม่มีออเดอร์ค้างอยู่ในระบบ
assert len(queue_system.active_orders) == 0

#แสดงผลการจำลองธุรกรรม
print(f"จำลองครบ {len(orders)} ออเดอร์ และส่งมอบทุกคิวเรียบร้อย")

จำลองครบ 300 ออเดอร์ และส่งมอบทุกคิวเรียบร้อย
